In [1]:
from pathlib import Path
import pandas as pd

# Folder containing clustering tables + CellProfiler exports
DATA_DIR = Path(r"C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime\datasets\for Danya ML")

# Show all files in the folder
files = sorted([p for p in DATA_DIR.iterdir() if p.is_file()])
for f in files:
    print(f.name)

clearing metadata for ML.ipynb
combined_clustered_data_1462_k5.csv
combined_clustered_data_1476_k2.csv
selected_intensity_variability_results.csv


In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime\datasets\for Danya ML")

tables = {}

for f in sorted(DATA_DIR.iterdir()):
    if not f.is_file():
        continue

    suffix = f.suffix.lower()

    try:
        if suffix == ".csv":
            tables[f.stem] = pd.read_csv(f)
            print(f"Loaded CSV:  {f.name} -> {tables[f.stem].shape}")

        elif suffix in [".xlsx", ".xls"]:
            tables[f.stem] = pd.read_excel(f)
            print(f"Loaded Excel: {f.name} -> {tables[f.stem].shape}")

    except Exception as e:
        print(f"Could not load {f.name}: {e}")

Loaded CSV:  combined_clustered_data_1462_k5.csv -> (410, 1971)
Loaded CSV:  combined_clustered_data_1476_k2.csv -> (667, 1971)
Loaded CSV:  selected_intensity_variability_results.csv -> (40, 5)


In [3]:
tables.keys()

dict_keys(['combined_clustered_data_1462_k5', 'combined_clustered_data_1476_k2', 'selected_intensity_variability_results'])

In [4]:
import numpy as np
import matplotlib.pyplot as plt


def plot_combined_surface_intensity_variability(
    surface_summaries,
    cmap_heatmap="coolwarm",
    cmap_std="Greys",
    vmin=-2.5,
    vmax=2.5,
    figsize=(22, 8),
    shorten_labels=False,
    show_heatmap_values=False,
    show_std_values=True,
    std_fmt="{:.3f}",
    label_fontsize=11,
    tick_fontsize=11,
    title_fontsize=20,
    cbar_fontsize=14,
    variability_scale="per_surface",  # "per_surface" or "global"
    cbar_label="Row-wise z-score of cluster mean intensity"
):
    """
    Plot two surfaces side by side, each with:
    - main heatmap: row-wise z-score of cluster means
    - side variability column: actual std across cluster means for each metric
    """

    if len(surface_summaries) != 2:
        raise ValueError("This plotting function expects exactly 2 surface summaries.")

    # Set global std range if requested
    if variability_scale == "global":
        all_std_vals = []
        for summary in surface_summaries:
            feats = summary["selected_feats"]
            vals = summary["variability_all"].loc[feats].to_numpy(dtype=float)
            all_std_vals.extend(vals)
        global_std_vmin = float(np.nanmin(all_std_vals))
        global_std_vmax = float(np.nanmax(all_std_vals))
    else:
        global_std_vmin = None
        global_std_vmax = None

    # Layout: heatmap1 | std1 | heatmap2 | std2 | colorbar
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(
        1, 5,
        width_ratios=[1.0, 0.22, 1.0, 0.22, 0.08]
    )

    ax_hm_1 = fig.add_subplot(gs[0, 0])
    ax_std_1 = fig.add_subplot(gs[0, 1], sharey=ax_hm_1)
    ax_hm_2 = fig.add_subplot(gs[0, 2])
    ax_std_2 = fig.add_subplot(gs[0, 3], sharey=ax_hm_2)
    cax = fig.add_subplot(gs[0, 4])

    heatmap_axes = [ax_hm_1, ax_hm_2]
    std_axes = [ax_std_1, ax_std_2]

    im_heat = None

    for idx, (ax_hm, ax_std, summary) in enumerate(zip(heatmap_axes, std_axes, surface_summaries)):
        heatmap_data = summary["heatmap_scaled"]
        row_labels = heatmap_data.index.tolist()
        col_labels = [f"C{c}" for c in heatmap_data.columns.tolist()]

        # Actual variability values for plotted rows
        std_vals = summary["variability_all"].loc[row_labels].to_numpy(dtype=float).reshape(-1, 1)

        if variability_scale == "global":
            std_vmin = global_std_vmin
            std_vmax = global_std_vmax
        else:
            std_vmin = float(np.nanmin(std_vals))
            std_vmax = float(np.nanmax(std_vals))

        if np.isclose(std_vmin, std_vmax):
            std_vmax = std_vmin + 1e-9

        # Main heatmap
        im_heat = ax_hm.imshow(
            heatmap_data.values,
            aspect="auto",
            cmap=cmap_heatmap,
            vmin=vmin,
            vmax=vmax
        )

        ax_hm.set_xticks(np.arange(len(col_labels)))
        ax_hm.set_xticklabels(col_labels, fontsize=tick_fontsize)
        ax_hm.set_xlabel("Cluster", fontsize=14)

        display_row_labels = []
        for feat in row_labels:
            feat_disp = shorten_intensity_label(feat) if shorten_labels else feat

            if feat in summary["top_feats"]:
                display_row_labels.append("↑ " + feat_disp)
            elif feat in summary["bottom_feats"]:
                display_row_labels.append("↓ " + feat_disp)
            else:
                display_row_labels.append(feat_disp)

        ax_hm.set_yticks(np.arange(len(display_row_labels)))
        ax_hm.set_yticklabels(display_row_labels, fontsize=label_fontsize)

        if idx == 0:
            ax_hm.yaxis.tick_left()
            ax_hm.tick_params(axis="y", labelleft=True, labelright=False, pad=4)
            for lab in ax_hm.get_yticklabels():
                lab.set_horizontalalignment("right")
        else:
            ax_hm.yaxis.tick_right()
            ax_hm.tick_params(axis="y", labelleft=False, labelright=True, pad=4)
            for lab in ax_hm.get_yticklabels():
                lab.set_horizontalalignment("left")

        n_top = len(summary["top_feats"])
        ax_hm.axhline(n_top - 0.5, color="black", linewidth=1.2)

        ax_hm.set_title(
            f"Surface {summary['surface_name']}\n"
            f"{len(summary['top_feats'])} most variable + {len(summary['bottom_feats'])} least variable\n"
            f"(variability across cluster means)",
            fontsize=title_fontsize
        )

        if show_heatmap_values:
            for i in range(heatmap_data.shape[0]):
                for j in range(heatmap_data.shape[1]):
                    ax_hm.text(
                        j, i,
                        f"{heatmap_data.values[i, j]:.2f}",
                        ha="center", va="center",
                        fontsize=7
                    )

        # Std side column
        ax_std.imshow(
            std_vals,
            aspect="auto",
            cmap=cmap_std,
            vmin=std_vmin,
            vmax=std_vmax
        )

        ax_std.set_xticks([0])
        ax_std.set_xticklabels(["std"], fontsize=tick_fontsize, rotation=90)
        ax_std.tick_params(axis="y", left=False, right=False, labelleft=False, labelright=False)

        ax_std.axhline(n_top - 0.5, color="black", linewidth=1.2)

        if show_std_values:
            std_norm = (std_vals.flatten() - std_vmin) / (std_vmax - std_vmin + 1e-12)
            for i, (val, norm_val) in enumerate(zip(std_vals.flatten(), std_norm)):
                text_color = "white" if norm_val > 0.6 else "black"
                ax_std.text(
                    0, i,
                    std_fmt.format(val),
                    ha="center", va="center",
                    fontsize=9,
                    color=text_color
                )

    cbar = fig.colorbar(im_heat, cax=cax)
    cbar.set_label(cbar_label, fontsize=cbar_fontsize)
    cbar.ax.tick_params(labelsize=tick_fontsize)

    fig.subplots_adjust(
        left=0.23,
        right=0.88,
        wspace=0.35
    )

    plt.show()

    return fig, {
        "heatmap_axes": heatmap_axes,
        "std_axes": std_axes,
        "colorbar_axis": cax
    }

In [5]:
fig, axes_dict = plot_combined_surface_intensity_variability(
    [results["1462"], results["1476"]],
    shorten_labels=False,
    show_std_values=True,
    variability_scale="global"
)

NameError: name 'results' is not defined

In [ ]:
export_table = pd.concat([
    pd.DataFrame({
        "surface": surface_name,
        "row_in_plot": range(1, len(summary["selected_feats"]) + 1),
        "metric_name": summary["selected_feats"],
        "group": [
            "most_variable" if f in summary["top_feats"] else "least_variable_nonzero"
            for f in summary["selected_feats"]
        ],
        "variability_std": [summary["variability_all"][f] for f in summary["selected_feats"]]
    })
    for surface_name, summary in [("1462", results["1462"]), ("1476", results["1476"])]
], ignore_index=True)

export_table.to_csv(r"C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime\datasets\for Danya ML\selected_intensity_variability_results.csv", index=False)
export_table.head()

,surface,row_in_plot,metric_name,group,variability_std
0,1462,1,phalloidin_Intensity_IntegratedIntensity_Phall...,most_variable,52.040154
1,1462,2,yap_Intensity_IntegratedIntensity_Phalloidin,most_variable,51.839841
2,1462,3,phalloidin_Intensity_IntegratedIntensity_YAP,most_variable,23.915685
3,1462,4,yap_Intensity_IntegratedIntensity_YAP,most_variable,23.820687
4,1462,5,phalloidin_Intensity_IntegratedIntensity_DNA,most_variable,17.729548


In [ ]:
import re
import pandas as pd


def explain_cp_metric(metric_name):
    """
    Convert a CellProfiler-style metric name into a short human-readable explanation.
    Expected style:
        <object>_Intensity_<measurement>_<channel>
    Example:
        dapi_Intensity_IntegratedIntensity_DNA
    """

    metric_name = str(metric_name)

    object_map = {
        "dapi": "segmented DAPI object",
        "phalloidin": "segmented phalloidin object",
        "yap": "segmented YAP object",
    }

    channel_map = {
        "DNA": "DNA channel",
        "Phalloidin": "phalloidin channel",
        "YAP": "YAP channel",
    }

    measurement_map = {
        "IntegratedIntensity": "sum of pixel intensities",
        "MeanIntensity": "mean pixel intensity",
        "StdIntensity": "standard deviation of pixel intensities",
        "MaxIntensity": "maximum pixel intensity",
        "MinIntensity": "minimum pixel intensity",
        "IntegratedIntensityEdge": "sum of edge-pixel intensities",
        "MeanIntensityEdge": "mean edge-pixel intensity",
        "StdIntensityEdge": "standard deviation of edge-pixel intensities",
        "MaxIntensityEdge": "maximum edge-pixel intensity",
        "MinIntensityEdge": "minimum edge-pixel intensity",
        "MassDisplacement": "distance between the binary-shape center and the intensity-weighted center",
        "LowerQuartileIntensity": "25th percentile of pixel intensities",
        "MedianIntensity": "median pixel intensity",
        "MADIntensity": "median absolute deviation of pixel intensities",
        "UpperQuartileIntensity": "75th percentile of pixel intensities",
        "Location_CenterMassIntensity_X": "X coordinate of the intensity-weighted center",
        "Location_CenterMassIntensity_Y": "Y coordinate of the intensity-weighted center",
        "Location_CenterMassIntensity_Z": "Z coordinate of the intensity-weighted center",
        "Location_MaxIntensity_X": "X coordinate of the brightest pixel",
        "Location_MaxIntensity_Y": "Y coordinate of the brightest pixel",
        "Location_MaxIntensity_Z": "Z coordinate of the brightest pixel",
    }

    # Support Percentile_N if present
    percentile_match = re.search(r"(^|_)Percentile_(\d+)(_|$)", metric_name)
    percentile_text = None
    if percentile_match:
        p = percentile_match.group(2)
        percentile_text = f"{p}th percentile of pixel intensities"

    parts = metric_name.split("_")
    obj_prefix = parts[0] if len(parts) > 0 else None
    obj_text = object_map.get(obj_prefix, f"object '{obj_prefix}'" if obj_prefix else "object")

    channel_text = None
    for token in reversed(parts):
        if token in channel_map:
            channel_text = channel_map[token]
            break

    measurement_text = None

    # First try exact known measurement keys
    for key, value in measurement_map.items():
        if key in metric_name:
            measurement_text = value
            break

    # Then try percentile
    if measurement_text is None and percentile_text is not None:
        measurement_text = percentile_text

    # Fallback if unknown
    if measurement_text is None:
        measurement_text = "intensity-derived measurement"

    if channel_text is not None:
        return f"{measurement_text} in the {channel_text}, measured within the {obj_text}"
    return f"{measurement_text} measured within the {obj_text}"


# Load your exported table
export_table = pd.read_csv(r"C:\Users\20212358\OneDrive - TU Eindhoven\pseudotime_GitHUB\pseudotime\datasets\for Danya ML\selected_intensity_variability_results.csv")

# Add explanation column
export_table["metric_explanation"] = export_table["metric_name"].apply(explain_cp_metric)

# Save updated file
export_table.to_csv("selected_intensity_variability_results_explained.csv", index=False)

# Preview
display(export_table.head(20))

,surface,row_in_plot,metric_name,group,variability_std,metric_explanation
0,1462,1,phalloidin_Intensity_IntegratedIntensity_Phall...,most_variable,52.040154,sum of pixel intensities in the phalloidin cha...
1,1462,2,yap_Intensity_IntegratedIntensity_Phalloidin,most_variable,51.839841,sum of pixel intensities in the phalloidin cha...
2,1462,3,phalloidin_Intensity_IntegratedIntensity_YAP,most_variable,23.915685,"sum of pixel intensities in the YAP channel, m..."
3,1462,4,yap_Intensity_IntegratedIntensity_YAP,most_variable,23.820687,"sum of pixel intensities in the YAP channel, m..."
4,1462,5,phalloidin_Intensity_IntegratedIntensity_DNA,most_variable,17.729548,"sum of pixel intensities in the DNA channel, m..."
5,1462,6,yap_Intensity_IntegratedIntensity_DNA,most_variable,17.558811,"sum of pixel intensities in the DNA channel, m..."
6,1462,7,dapi_Intensity_IntegratedIntensity_DNA,most_variable,9.578168,"sum of pixel intensities in the DNA channel, m..."
7,1462,8,phalloidin_Location_MaxIntensity_X_Phalloidin,most_variable,8.387288,maximum pixel intensity in the phalloidin chan...
8,1462,9,yap_Location_MaxIntensity_X_Phalloidin,most_variable,7.865266,maximum pixel intensity in the phalloidin chan...
9,1462,10,dapi_Location_MaxIntensity_X_Phalloidin,most_variable,6.224292,maximum pixel intensity in the phalloidin chan...
